In [17]:
# function _mats_to_mt(mats::Vector{<:AbstractMatrix{<:Integer}})::Array{Int,3}
#     r = length(mats)
#     r ≥ 1 || error("_mats_to_mt: empty list of matrices")
#     mt = zeros(Int, r, r, r)
#     @inbounds for a in 1:r
#         A = mats[a]
#         size(A,1) == r && size(A,2) == r || error("_mats_to_mt: mat $a has wrong size $(size(A)) (expected $r×$r)")
#         mt[ :, a, : ] .= A
#     end
#     return mt
# end

function _mats_to_mt(mats)
    r = length(mats)
    [ mats[i][j,k] for i in 1:r, j in 1:r, k in 1:r  ]
end

_mats_to_mt (generic function with 2 methods)

In [40]:
function _son2_rules_odd(m::Integer)
    isodd(m) || throw(ArgumentError("_son2_rules_odd expects odd N, got N=$m"))
    m ≥ 5    || throw(ArgumentError("_son2_rules_odd expects N≥5 (odd), got N=$m"))

    r    = (m - 1) ÷ 2
    rank = (m + 7) ÷ 2 

    # convenience
    ar(i) = _e(i, rank)

    # mat1 = IdentityMatrix[rank]
    mat1 = zeros(Int, rank, rank)
    for i in 1:rank
        mat1[i,i] = 1
    end

    print(mat1)
    # matZ = Table[ Which[...], {i,rank} ]
    matZ = zeros(Int, rank, rank) 
    @inbounds for i in 1:rank
        v = if i == 1
            ar(2)
        elseif i == 2
            ar(1)
        elseif 3 <= i <= 4
            ar(3 + mod(i, 2))   # i=3 -> 4, i=4 -> 3
        else
            ar(i)
        end
        matZ[i, :] .= v
    end

    # matXe1 = Table[ Which[...], {i,rank} ]
    matXe1 = zeros(Int, rank, rank)
    @inbounds for i in 1:rank
        v = if i == 1
            ar(3)
        elseif i == 2
            ar(4)
        elseif i == 3
            ar(1) + sum( ar(j) for j ∈ 5:rank )
        elseif i == 4
            ar(2) + sum( ar(j) for j ∈ 5:rank )
        else
            ar(3) + ar(4)
        end
        matXe1[i, :] .= v
    end

    # matXe2 = Table[ Which[...], {i,rank} ]
    matXe2 = zeros(Int, rank, rank)
    @inbounds for i in 1:rank
        v = if i == 1
            ar(4)
        elseif i == 2
            ar(3)
        elseif i == 3
            ar(2) + sum( ar(j) for j ∈ 5:rank )
        elseif i == 4
            ar(1) + sum( ar(j) for j ∈ 5:rank )
        else
            ar(3) + ar(4)
        end
        matXe2[i, :] .= v
    end

    # matY[j_] := Table[ Which[...], {i,rank} ]
    function matY(j::Int)::Matrix{Int}
        M = zeros(Int, rank, rank)
        @inbounds for i in 1:rank
            v = if i == 1
                ar(j + 4)
            elseif i == 2
                ar(j + 4)
            elseif i == 3
                ar(3) + ar(4)
            elseif i == 4
                ar(3) + ar(4)
            else
                ii = i - 4
                if ii == j
                    # ar[1] + ar[2] + ar[ Min[2j, m-2j] + 4 ]
                    t = min(2*j, m - 2*j) + 4
                    ar(1) + ar(2) + ar(t)
                else
                    # ar[ Abs[ii-j] + 4 ] + ar[ Min[ii+j, m-ii-j+4] + 4 ]
                    t1 = abs(ii - j) + 4
                    t2 = min(ii + j, m - ii - j) + 4
                    ar(t1) + ar(t2)
                end
            end
            M[i, :] .= v
        end
        return M
    end

    #   Transpose /@ Join[{mat1, matZ, matXe1, matXe2}, matY /@ Range[r]]
    mats = Matrix{Int}[]
    push!( mats, mat1 )
    push!( mats, matZ )
    push!( mats, matXe1 )
    push!( mats, matXe2 )
    for j in 1:r
        push!( mats, matY(j) )
    end

    # Convert fusion matrices to multiplication table
    return mats
end
@inline function _e(i::Int, rank::Int)::Vector{Int}
    v = zeros(Int, rank)
    v[i] = 1
    return v
end

_e (generic function with 1 method)

In [41]:
mt = _son2_rules_odd(5)

[1 0 0 0 0 0; 0 1 0 0 0 0; 0 0 1 0 0 0; 0 0 0 1 0 0; 0 0 0 0 1 0; 0 0 0 0 0 1]

6-element Vector{Matrix{Int64}}:
 [1 0 … 0 0; 0 1 … 0 0; … ; 0 0 … 1 0; 0 0 … 0 1]
 [0 1 … 0 0; 1 0 … 0 0; … ; 0 0 … 1 0; 0 0 … 0 1]
 [0 0 … 0 0; 0 0 … 0 0; … ; 0 0 … 0 0; 0 0 … 0 0]
 [0 0 … 0 0; 0 0 … 0 0; … ; 0 0 … 0 0; 0 0 … 0 0]
 [0 0 … 1 0; 0 0 … 1 0; … ; 1 1 … 0 1; 0 0 … 1 1]
 [0 0 … 0 1; 0 0 … 0 1; … ; 0 0 … 1 1; 1 1 … 1 0]

In [45]:
amt = [ mt[i][j,k] for i in 1:6, j in 1:6, k in 1:6 ]

6×6×6 Array{Int64, 3}:
[:, :, 1] =
 1  0  0  0  0  0
 0  1  0  0  0  0
 0  0  1  0  0  0
 0  0  0  1  0  0
 0  0  0  0  1  0
 0  0  0  0  0  1

[:, :, 2] =
 0  1  0  0  0  0
 1  0  0  0  0  0
 0  0  0  1  0  0
 0  0  1  0  0  0
 0  0  0  0  1  0
 0  0  0  0  0  1

[:, :, 3] =
 0  0  1  0  0  0
 0  0  0  1  0  0
 1  0  0  0  1  1
 0  1  0  0  1  1
 0  0  1  1  0  0
 0  0  1  1  0  0

[:, :, 4] =
 0  0  0  1  0  0
 0  0  1  0  0  0
 0  1  0  0  1  1
 1  0  0  0  1  1
 0  0  1  1  0  0
 0  0  1  1  0  0

[:, :, 5] =
 0  0  0  0  1  0
 0  0  0  0  1  0
 0  0  1  1  0  0
 0  0  1  1  0  0
 1  1  0  0  0  1
 0  0  0  0  1  1

[:, :, 6] =
 0  0  0  0  0  1
 0  0  0  0  0  1
 0  0  1  1  0  0
 0  0  1  1  0  0
 0  0  0  0  1  1
 1  1  0  0  1  0

In [46]:
fusion_ring(amt)

UndefVarError: UndefVarError: `fusion_ring` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [71]:
@inline 𝟙  = 1
@inline Θ  = 2
@inline Φ₁ = 3
@inline Φ₂ = 4
@inline σ₁ = 5
@inline σ₂ = 6
@inline τ₁ = 7
@inline τ₂ = 8
@inline Φ(j::Int) = 8 + j

function _son2_rules_div2(p::Integer)
    p ≥ 1 || throw(ArgumentError("_son2_rules_div2 expects p≥1, got p=$p"))
    rank = p + 7
    maxphi = rank - 8  # = p-1

    ar(i) = _e(i, rank)

    # sums: sumEvenΛs = Σ_{i=2,4,...,p-1} Φ[i], sumOddΛs = Σ_{i=1,3,...,p-1} Φ[i]
    sumEven = zeros(Int, rank)
    sumOdd  = zeros(Int, rank)
    for i in 1:maxphi
        (isodd(i) ? (sumOdd[Φ(i)] += 1) : (sumEven[Φ(i)] += 1))
    end

    mats = Matrix{Int}[]

    # matId = IdentityMatrix[rank]
    matId = zeros(Int, rank, rank)
    for i ∈ 1:rank 
        matId[i,i] = 1
    end

    push!(mats, matId)

    # matΘ
    matTh = zeros(Int, rank, rank)
    @inbounds for i in 1:rank
        v = if i == 𝟙
            ar(Θ)
        elseif i == Θ
            ar(𝟙)
        elseif i == Φ₁
            ar(Φ₂)
        elseif i == Φ₂
            ar(Φ₁)
        elseif i == σ₁
            ar(τ₁)
        elseif i == σ₂
            ar(τ₂)
        elseif i == τ₁
            ar(σ₁)
        elseif i == τ₂
            ar(σ₂)
        else
            ar(i) # Φ-lambdas fixed
        end
        matTh[i, :] .= v
    end
    push!(mats, matTh)

    # matΦ1
    matPhi1 = zeros(Int, rank, rank)
    @inbounds for i in 1:rank
        v = if i == 𝟙
            ar(Φ₁)
        elseif i == Θ
            ar(Φ₂)
        elseif i == Φ₁
            ar(Θ)
        elseif i == Φ₂
            ar(𝟙)
        elseif i == σ₁
            ar(σ₂)
        elseif i == σ₂
            ar(τ₁)
        elseif i == τ₁
            ar(τ₂)
        elseif i == τ₂
            ar(σ₁)
        else
            # Φ[p - (i-8)]
            j = i - 8
            ar(Φ(p - j))
        end
        matPhi1[i, :] .= v
    end
    push!( mats, matPhi1 )

    # matΦ2
    matPhi2 = zeros(Int, rank, rank)
    @inbounds for i in 1:rank
        v = if i == 𝟙
            ar(Φ₂)
        elseif i == Θ
            ar(Φ₁)
        elseif i == Φ₁
            ar(𝟙)
        elseif i == Φ₂
            ar(Θ)
        elseif i == σ₁
            ar(τ₂)
        elseif i == σ₂
            ar(σ₁)
        elseif i == τ₁
            ar(σ₂)
        elseif i == τ₂
            ar(τ₁)
        else
            j = i - 8
            ar( Φ(p - j) )
        end
        matPhi2[i, :] .= v
    end
    push!( mats, matPhi2 )

    # matσ1
    matSig1 = zeros(Int, rank, rank)
    @inbounds for i in 1:rank
        v = if i == 𝟙
            ar(σ₁)
        elseif i == Θ
            ar(τ₁)
        elseif i == Φ₁
            ar(σ₂)
        elseif i == Φ₂
            ar(τ₂)
        elseif i == σ₁
            ar(Φ₂) + sumOdd
        elseif i == σ₂
            ar(𝟙)  + sumEven
        elseif i == τ₁
            ar(Φ₁) + sumOdd
        elseif i == τ₂
            ar(Θ)  + sumEven
        else
            # If[OddQ[i], σ2+τ2, σ1+τ1]
            isodd(i) ? ar(σ₂) + ar(τ₂) : ar(σ₁) + ar(τ₁)
        end
        matSig1[i, :] .= v
    end
    push!( mats, matSig1 )

    # matσ2
    matSig2 = zeros(Int, rank, rank)
    @inbounds for i in 1:rank
        v = if i == 𝟙
            ar(σ₂)
        elseif i == Θ
            ar(τ₂)
        elseif i == Φ₁
            ar(τ₁)
        elseif i == Φ₂
            ar(σ₁)
        elseif i == σ₁
            ar(𝟙)  + sumEven
        elseif i == σ₂
            ar(Φ₁) + sumOdd
        elseif i == τ₁
            ar(Θ)  + sumEven
        elseif i == τ₂
            ar(Φ₂) + sumOdd
        else
            # If[EvenQ[i], σ2+τ2, σ1+τ1]
            iseven(i) ? ar(σ₂) + ar(τ₂) : ar(σ₁) + ar(τ₁)
        end
        matSig2[i, :] .= v
    end
    push!( mats, matSig2 )

    # matτ1
    matTau1 = zeros(Int, rank, rank)
    @inbounds for i in 1:rank
        v = if i == 𝟙
            ar(τ₁)
        elseif i == Θ
            ar(σ₁)
        elseif i == Φ₁
            ar(τ₂)
        elseif i == Φ₂
            ar(σ₂)
        elseif i == σ₁
            ar(Φ₁) + sumOdd
        elseif i == σ₂
            ar(Θ)  + sumEven
        elseif i == τ₁
            ar(Φ₂) + sumOdd
        elseif i == τ₂
            ar(𝟙)  + sumEven
        else
            isodd(i) ? ar(σ₂) + ar(τ₂) : ar(σ₁) + ar(τ₁)
        end
        matTau1[i, :] .= v
    end
    push!(mats, matTau1)

    # matτ2
    matTau2 = zeros(Int, rank, rank)
    @inbounds for i in 1:rank
        v = if i == 𝟙
            ar(τ₂)
        elseif i == Θ
            ar(σ₂)
        elseif i == Φ₁
            ar(σ₁)
        elseif i == Φ₂
            ar(τ₁)
        elseif i == σ₁
            ar(Θ)  + sumEven
        elseif i == σ₂
            ar(Φ₂) + sumOdd
        elseif i == τ₁
            ar(𝟙)  + sumEven
        elseif i == τ₂
            ar(Φ₁) + sumOdd
        else
            iseven(i) ? ar(σ₂) + ar(τ₂) : ar(σ₁) .+ ar(τ₁) 
        end
        matTau2[i, :] .= v
    end
    push!(mats, matTau2)

    # matΦ[j] for j = 1..(rank-8)
    function matPhi(j::Int)::Matrix{Int}
        M = zeros(Int, rank, rank)
        @inbounds for i in 1:rank
            v = if i == 𝟙
                ar(Φ(j))
            elseif i == Θ
                ar(Φ(j))
            elseif i == Φ₁
                ar(Φ(p - j))
            elseif i == Φ₂
                ar(Φ(p - j))
            elseif i == σ₁
                isodd(j)  ? ar(σ₂) + ar(τ₂) : ar(σ₁) + ar(τ₁)
            elseif i == σ₂
                iseven(j) ? ar(σ₂) + ar(τ₂) : ar(σ₁) + ar(τ₁)
            elseif i == τ₁
                isodd(j)  ? ar(σ₂) + ar(τ₂) : ar(σ₁) + ar(τ₁)
            elseif i == τ₂
                iseven(j) ? ar(σ₂) + ar(τ₂) : ar(σ₁) + ar(τ₁)
            else
                ii = i - 8
                if ii == j && (2*j < p)
                    ar(𝟙) + ar(Θ) + ar( Φ(2 * j) )
                elseif ii == j && (2*j > p)
                    ar(𝟙) + ar(Θ) + ar( Φ(2 * (p - j)) )
                elseif ii + j < p
                    ar( Φ(abs(ii - j)) ) + ar( Φ(ii + j) )
                elseif ii + j > p
                    ar( Φ(abs(ii - j)) ) + ar( Φ(2*p - ii - j) )
                else
                    # ii == p - j
                    ar(Φ₁) + ar(Φ₂) + ar( Φ( abs(p - 2*ii) ) )
                end
            end
            M[i, :] .= v
        end
        return M
    end

    for j in 1:maxphi
        push!(mats, transpose(matPhi(j)))
    end

    _mats_to_mt(mats)
end

_son2_rules_div2 (generic function with 1 method)

In [75]:
using Oscar
G_perm = @permutation_group( 5, (2,5)(3,4), (1,3)(4,5) )

Permutation group of degree 5

In [76]:
order(G_perm)

10

In [77]:
collect(G_perm)

10-element Vector{PermGroupElem}:
 ()
 (2,5)(3,4)
 (1,4,2,5,3)
 (1,4)(2,3)
 (1,2,3,4,5)
 (1,2)(3,5)
 (1,5,4,3,2)
 (1,5)(2,4)
 (1,3,5,2,4)
 (1,3)(4,5)

In [95]:
function cayley_table( grp::Group )::Matrix{Int64}
    els = collect(grp)
    r   = order(grp)
    [ findfirst( x -> x == g1 * g2, els)  for g1 in els, g2 in els  ]
end

cayley_table (generic function with 2 methods)

In [94]:
G_perm |> typeof |> supertype |> supertype

Group

In [99]:
G_perm |> order |> Int64 |> typeof

Int64